# 6.7: GPUs

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

## 6.7.1: Computing Devices

In [ ]:
def cpu(): #@save
    """Get the SPU device."""
    return torch.device('cpu')

def gpu(i=0): #@save
    """Get a GPU device"""
    return torch.device(f'cuda:{i}')

cpu(), gpu(), gpu(1) # gpu(1) is meaningful if you have one extra

(device(type='cpu'),
 device(type='cuda', index=0),
 device(type='cuda', index=1))

In [5]:
def num_gpus(): #@save
    """Get the number of available GPUs."""
    return torch.cuda.device_count()

num_gpus()

0

In [9]:
def try_gpu(i=0): #@save
    """Return gpu(i) if it exists, else return cpu()"""
    if num_gpus() >= i + 1:
        return gpu(i)
    return cpu()

def try_all_gpus(): #@save
    """`Return all available GPUs, or [cpu(),] if no GPU exists"""
    return [gpu(i) for i in range(num_gpus())]

try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cpu'), device(type='cpu'), [])

## 6.7.2: Tensors and GPUs

In [ ]:
x = torch.tensor([1,2,3])
x.device

device(type='cpu')

Note that, in order to operate on multiple terms, they need to be on the same device.

### Storage on the GPU

In [11]:
X = torch.ones(2,3, device=try_gpu())
X

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [12]:
Y = torch.rand(2,3,device=try_gpu(1))
Y

tensor([[0.5851, 0.7665, 0.6045],
        [0.7652, 0.0688, 0.5985]])

### Copying

In [13]:
# Move X to Y's device via copying so we can perform the operation
Z = X.cuda(1)
print(X)
print(Z)

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
Y+Z

Note: moving a tensor to the device it is already on will return the original device, instead of allocating new memory.

## 6.7.3: Neural Networks and GPUs

In [14]:
net = nn.Sequential(nn.LazyLinear(1))
net = net.to(device=try_gpu())

c:\Users\Py Torch\.conda\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [15]:
# when the input is a tensor on the GPU, the model will calculate the same result on the same GPU
net(X)

tensor([[-0.3514],
        [-0.3514]], grad_fn=<AddmmBackward0>)

In [16]:
net[0].weight.data.device

device(type='cpu')

Let's add the trainer support GPU.

In [17]:
@d2l.add_to_class(d2l.Trainer) #@save
def __init__(self, max_epochs, num_gpus=0, gradient_clip_val=0):
    self.save_hyperparameters()
    self.gpus = [d2l.gpu(i) for i in range(min(num_gpus(), d2l.num_gpus()))]

@d2l.add_to_class(d2l.Trainer) #@save
def prepare_batch(self, batch):
    if self.gpus:
        batch = [a.to(self.gpus[0]) for a in batch]
    return batch

@d2l.add_to_class(d2l.Trainer) #@save
def prepare_model(self, model):
    model.trainer = self
    model.board.xlim = [0, self.max_epochs]
    if self.gpus:
        model.to(self.gpus[0])
    self.model = model

In short, as long as all data and parameters are on the same device, we can learn models efficiently.

# Summary:

We can specify devices for storage and calculation, such as the CPU or GPU. By default, data is created in the main memory and then uses the CPU for calculations. The deep learning framework requires all input data for calculation to be on the same device, be it CPU or the same GPU. You can lose significant performance by moving data without care. A typical mistake is as follows: computing the loss for every minibatch on the GPU and reporting it back to the user on the command line (or logging it in a NumPy ndarray) will trigger a global interpreter lock which stalls all GPUs. It is much better to allocate memory for logging inside the GPU and only move larger logs.